# Lab 2.1 · Data profiling และทำความสะอาดข้อมูล (บ้านบรู)
**ขั้นตอน:** โหลด → profiling (รายงานปัญหาก่อน ยังไม่แก้) → ทำความสะอาด → ตรวจ → export `sales_clean.csv`

ใน Colab: อัปโหลด `sales_raw.csv` หรือ `sales_raw.xlsx` และ `products.xlsx` ผ่านแถบ Files ด้านซ้าย แล้วกด Run all

In [ ]:
import pandas as pd, re, os
RAW = "sales_raw.csv" if os.path.exists("sales_raw.csv") else "sales_raw.xlsx"
read = pd.read_csv if RAW.endswith(".csv") else pd.read_excel
df = read(RAW, dtype=str)          # อ่านทุกคอลัมน์เป็นข้อความก่อน จะได้เห็นค่าผิดรูปแบบ
products = pd.read_excel("products.xlsx", dtype=str) if os.path.exists("products.xlsx") else pd.read_csv("products.csv", dtype=str)
print(RAW, df.shape)
df.head()

## 1) Profiling: รายงานปัญหาก่อน ยังไม่แก้ข้อมูล

In [ ]:
profile = pd.DataFrame({"null": df.isna().sum(), "unique": df.nunique()})
profile

In [ ]:
dt = df["datetime"].astype(str)
is_iso   = dt.str.match(r"^20\d\d-\d\d-\d\dT")
is_be    = dt.str.match(r"^25\d\d-\d\d-\d\dT")              # ปี พ.ศ.
is_dmy   = dt.str.match(r"^\d\d/\d\d/\d{4}")                 # 01/04/2025 10:54
is_excel = dt.str.match(r"^20\d\d-\d\d-\d\d \d\d:\d\d:\d\d$")  # Excel แปลง DD/MM เป็น MM/DD ให้เอง (เฉพาะไฟล์ .xlsx)
price_num = pd.to_numeric(df["unit_price"], errors="coerce")
qty_num   = pd.to_numeric(df["qty"], errors="coerce")
STD_BRANCH = {"สยาม", "สีลม", "อารีย์", "บางนา", "มหาวิทยาลัย"}

issues = pd.DataFrame([
    ("แถวซ้ำทุกคอลัมน์",            df.duplicated().sum()),
    ("วันที่เป็นปี พ.ศ.",            is_be.sum()),
    ("วันที่แบบ DD/MM/YYYY (ข้อความ)", is_dmy.sum()),
    ("วันที่ที่ Excel แปลงแล้ว (สลับวัน/เดือน)", is_excel.sum()),
    ("ชื่อสาขาไม่มาตรฐาน",          (~df["branch"].isin(STD_BRANCH)).sum()),
    ("ราคาเป็นข้อความ เช่น '65.00 บาท'", price_num.isna().sum()),
    ("ราคาติดลบ",                   (price_num < 0).sum()),
    ("qty = 0",                     (qty_num == 0).sum()),
    ("product_id ว่าง",             df["product_id"].isna().sum()),
], columns=["ปัญหา", "จำนวนแถว"])
issues

In [ ]:
df["branch"].value_counts()

## 2) การตัดสินใจทำความสะอาด (บันทึกเหตุผลทุกข้อ)
| ปัญหา | ตัดสินใจ | เหตุผล |
|---|---|---|
| แถวซ้ำ | ลบ `drop_duplicates()` **ทุกคอลัมน์** | POS ส่งซ้ำ · **ห้ามใช้ `subset='order_id'`** เพราะ 1 บิลมีหลายแถว จะลบแถวดีทิ้ง |
| ปี พ.ศ. | ลบ 543 | รูปแบบอื่นเหมือน ISO ทุกอย่าง |
| DD/MM/YYYY | แปลงด้วย `dayfirst=True` เป็นเวลาไทย | รูปแบบวันที่แบบไทย |
| Excel สลับวัน/เดือน | สลับกลับ | เกิดตอนแปลงไฟล์เป็น .xlsx (ไม่มีในไฟล์ .csv) · ตรวจแล้ววันที่ตกนอกช่วงข้อมูลถ้าไม่สลับ |
| ชื่อสาขา | map เป็นชื่อมาตรฐาน 5 สาขา | "ม." / "มหาลัย" = มหาวิทยาลัย (*ควรยืนยันกับทีม POS*) |
| ราคาเป็นข้อความ | ตัด "บาท" แล้วแปลงเป็นตัวเลข | |
| ราคาติดลบ | แปลงเป็นบวก | ตัวเลขตรงกับราคาเมนู และบิลไม่มีรายการคืนเงินคู่กัน → น่าจะพิมพ์ผิด (*ควรยืนยันกับฝ่ายบัญชี*) |
| qty = 0 | ลบแถว | ไม่รู้ว่าขายจริงกี่ชิ้น อาจเป็นบิลยกเลิก (*ควรถามผู้จัดการสาขา*) |
| product_id ว่าง | ลบแถว | ไม่รู้ว่าเป็นเมนูไหน ราคาเดียวกันมีหลายเมนู เดาไม่ได้ |

In [ ]:
TH = 7  # เวลาไทย +07:00
def fix_datetime(s):
    s = str(s).strip()
    m = re.match(r"^(\d\d)/(\d\d)/(\d{4})[ T](\d\d):(\d\d)(?::(\d\d))?", s)
    if m:                                                        # DD/MM/YYYY HH:MM (ปีอาจเป็น พ.ศ.)
        d, mo, y, h, mi, sec = m.groups()
        y = int(y) - 543 if int(y) > 2400 else int(y)
        return f"{y}-{mo}-{d}T{h}:{mi}:{sec or '00'}+07:00"
    if re.match(r"^25\d\d-", s):                             # พ.ศ. → ค.ศ. แล้วตรวจรูปแบบต่อ
        s = str(int(s[:4]) - 543) + s[4:]
    m = re.match(r"^(\d{4})-(\d\d)-(\d\d) (\d\d):(\d\d):(\d\d)$", s)
    if m:                                                        # Excel สลับ: เดือนจริงอยู่ในช่องวัน
        y, a, b, h, mi, sec = m.groups()
        return f"{y}-{b}-{a}T{h}:{mi}:{sec}+07:00"
    return s

BRANCH_MAP = {"Siam": "สยาม", "สาขาสยาม": "สยาม", "Silom": "สีลม", "Ari": "อารีย์", "อารีย": "อารีย์",
              "Bangna": "บางนา", "ม.": "มหาวิทยาลัย", "มหาลัย": "มหาวิทยาลัย"}

log = []
clean = df.copy()
n0 = len(clean)
clean = clean.drop_duplicates();                                        log.append(("ลบแถวซ้ำ", n0 - len(clean)))
clean["datetime"] = clean["datetime"].map(fix_datetime)
clean["branch"] = clean["branch"].str.strip().replace(BRANCH_MAP)
clean["unit_price"] = pd.to_numeric(clean["unit_price"].str.replace("บาท", "").str.strip())
log.append(("ราคาติดลบ → เป็นบวก (ไม่ลบแถว)", int((clean["unit_price"] < 0).sum())))
clean["unit_price"] = clean["unit_price"].abs()
clean["qty"] = pd.to_numeric(clean["qty"])
n = len(clean); clean = clean[clean["qty"] > 0];                         log.append(("ลบ qty = 0", n - len(clean)))
n = len(clean); clean = clean[clean["product_id"].notna()];              log.append(("ลบ product_id ว่าง", n - len(clean)))
clean["unit_price"] = clean["unit_price"].astype(int)
clean["qty"] = clean["qty"].astype(int)
pd.DataFrame(log, columns=["ขั้นตอน", "จำนวนแถว"])

## 3) ตรวจผล (ทุกข้อต้องเป็น True)

In [ ]:
checks = {
    "วันที่เป็น ISO ทุกแถว":           clean["datetime"].str.match(r"^20\d\d-\d\d-\d\dT\d\d:\d\d:\d\d\+07:00$").all(),
    "วันที่อยู่ในช่วง 1 เม.ย. 68–20 ก.ย. 69": clean["datetime"].str[:10].between("2025-04-01", "2026-09-20").all(),
    "มีแค่ 5 สาขามาตรฐาน":            set(clean["branch"]) == STD_BRANCH,
    "ราคา > 0":                       (clean["unit_price"] > 0).all(),
    "qty > 0":                        (clean["qty"] > 0).all(),
    "ไม่มีแถวซ้ำ":                    not clean.duplicated().any(),
    "product_id อยู่ใน products ทุกแถว": clean["product_id"].isin(products["product_id"]).all(),
}
checks

In [ ]:
clean["revenue"] = clean["qty"] * clean["unit_price"]
print("แถวก่อน", len(df), "→ หลัง", len(clean), "(ตัด", len(df) - len(clean), "แถว)")
print("ยอดขายรวม ฿{:,.0f} · บิล {:,}".format(clean["revenue"].sum(), clean["order_id"].nunique()))
clean.groupby("branch")["revenue"].sum().sort_values(ascending=False)

In [ ]:
clean.drop(columns="revenue").to_csv("sales_clean.csv", index=False)
print("บันทึก sales_clean.csv แล้ว")